In [9]:
import torch
from torch.nn import Sequential, Conv2d, ReLU, Linear, Flatten, Sigmoid
from zennit.attribution import Gradient

from torch.nn import functional as F

In [10]:
torch.manual_seed(1)

input = torch.randn(5)

model = Sigmoid()

In [11]:
# Approach 1: compute the gradient analytically
@torch.no_grad()
def grad_sigmoid(x):
    return F.sigmoid(x) * (1- F.sigmoid(x))

grad_sigmoid(input)

tensor([0.2245, 0.2456, 0.2498, 0.2273, 0.2377])

In [15]:
# Approach 2 compute the gradient from Zennit
in1 = input.clone()

with Gradient(model) as attributor:
    output, relevance = attributor(in1, lambda a: torch.ones_like(a))
    
relevance

tensor([0.2245, 0.2456, 0.2498, 0.2273, 0.2377])

In [16]:
# Approach 3:  compute gradient using PyTorch
in2 = input.clone()
in2.requires_grad_(True)
in2.retain_grad()

with torch.no_grad():
    assert torch.allclose(in2, in1)
    
output2 = model(in2)
output2.sum().backward()

assert torch.allclose(output2, output)

(in2.grad)

tensor([0.2245, 0.2456, 0.2498, 0.2273, 0.2377])